#### 0. Global plotting style settings
    - remove top/right spines
    - black axes
    - dpi=600
    - save with transparent background

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import zscore

# ======Nature color palette dictionary (used consistently across the entire project)=======
NATURE_PALETTE = {
    'R':  '#A8C8E8',
    'CO': '#F5A8A0',
    'CT': '#1F3A6E',
    'CR': '#8B1A1A'
}
GROUP_ORDER = ['R', 'CO', 'CT', 'CR']  # Biological logical order

def apply_nature_style(ax):
    """
    Apply Nature-style axis formatting consistently:
    - Remove top and right spines
    - Keep left and bottom spines
    - set color to black
    Parameters: ax is a matplotlib Axes object (the coordinate axes of a plot)
    """
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('black')
    ax.spines['bottom'].set_color('black')
    ax.tick_params(colors='black')
    return ax

def save_figure(fig, filepath):
    """
    Standardized saving settings: dpi=600,
    transparent=True (transparent background)
    """
    fig.savefig(filepath, dpi=600, transparent=True)
    print(f"Saved figure to {filepath}")

# Function to extract sample groups using regular expressions
def extract_group(sample_name):
    """
    Extract group name 'CR' from sample names like 'CR2' (remove trailing digits)
    """
    return re.sub(r'\d+$', '', sample_name)

print("# Plotting style settings loaded successfully ✅")


#### 1. sns.heatmap
     Calculate pairwise correlations among 12 samples using the real gene_fpkm.csv, to validate CR2 as an outlier

In [ ]:
# ===========Read my transcriptome data==============
df = pd.read_csv("D:/Python-learning/my_transcriptome_data/gene_fpkm.csv")
print("Data shape: ", df.shape)
df.head()

# =========Seperate Dataframe============
sample_cols = ['R1', 'R2', 'R3', 'CO1', 'CO2', 'CO3',
               'CT1', 'CT2', 'CT3', 'CR1', 'CR2', 'CR3']
expr_df = df[['gene_id']+sample_cols].copy()

# ===================Annotation Dataframe=======================
annot_cols = ['NR Description', 'NT Description', 'Swissprot Description',
              'PFAM description', 'BP Description', 'MF Description', 'CC Description']
annot_df = df[['gene_id']+annot_cols].copy()

print(f"expression columns: {expr_df.shape}")
print(f"annotation columns: {annot_df.shape}")
display(expr_df.head())

#### 2. Grouped boxplot with stripplot overlay

- select top 6 genes by |log2FC| (CR vs R)
- boxplot shows distribution summary per condition
- stripplot overlay shows individual replicate points
- validate whether n=3 replicates are tightly clustered or scattered

In [ ]:
# ============Caluculate log2 Fold Change (CR vs R) for each gene=================

# Define sample columes belongs to each group
R_cols = ['R1', 'R2', 'R3']
CR_cols = ['CR1', 'CR2', 'CR3']

# Calculate the mean expression per group for each gene
mean_R = expr_df[R_cols].mean(axis=1)
mean_CR = expr_df[CR_cols].mean(axis=1)

# Calculate log2 fold change, adding pseudocount +1 to avoid log2(0) error
log2FC = np.log2((mean_CR + 1)/(mean_R + 1))

# Attatched log2FC as new colume to a working copy of expr_df
expr_df_fc = expr_df.copy()
expr_df_fc['log2FC'] = log2FC

# Sort by absolute log2FC value
expr_df_fc['abs_log2FC'] = expr_df_fc['log2FC'].abs()
top6_genes = expr_df_fc.sort_values('abs_log2FC', ascending=False).head(6)

# Display the selected top6 genes and their FC values
print("Top 6 genes with largest CR vs R FC values")
print(top6_genes[['gene_id', 'log2FC']])

In [ ]:
# =========Step 2a: Reshap top 6 genes from wide formate to long formate===========
#get the gene_id list of the top 6 genes
top6_ids = top6_genes['gene_id'].tolist()

# subset the expression matrix
top6_expr = expr_df[expr_df['gene_id'].isin(top6_ids)][['gene_id']+sample_cols]

# reshape wide to long
top6_long = top6_expr.melt(id_vars='gene_id', var_name='sample', value_name='expression')

top6_long['condition'] = top6_long['sample'].apply(extract_group)

top6_long['condition'] = pd.Categorical(top6_long['condition'], categories=GROUP_ORDER, ordered=True)

top6_long['log2_expression'] = np.log2(top6_long['expression'] + 1)

print(top6_long.shape)
top6_long.head(12)

In [ ]:
# ============Step 2b: Grouped boxplot + stripplot, one subplot per gene=============

fig, axes = plt.subplots(2, 3, figsize=(15, 8))    #2*3 grid for 6 genes
axes = axes.flatten()    #flaltten 2D grid into 1D list for easy looping

for i, gene_id in enumerate(top6_ids):
    ax = axes[i]
    gene_data = top6_long[top6_long['gene_id'] == gene_id]    # subset one gene's data

    # boxplot: distribution summary per condition
    sns.boxplot(data=gene_data, x='condition', y='log2_expression',
                order=GROUP_ORDER, hue='condition', palette=NATURE_PALETTE, legend=False, ax=ax, showfliers=False)

    # stripplot overlay
    sns.stripplot(data=gene_data, x='condition', y='log2_expression',
                  order=GROUP_ORDER, color='black', size=5, jitter=True, ax=ax)

    ax.set_title(gene_id, fontsize=10, color='black')
    ax.set_xlabel('')
    ax.set_ylabel('log2(FPKM + 1)', color='black')
    apply_nature_style(ax)

plt.tight_layout()
plt.show()

save_figure(fig, r"D:\Python-learning\week02\figures\fig2_boxplot_stripplot.png")

In [ ]:
# ===============Step 3a: Prepare data and standardize for PCA==================

expr_matrix = expr_df.set_index('gene_id')[sample_cols]   #set gene_idas index for reference

expr_matrix_T = expr_matrix.T    # transpose: rows=genes -> rows=samples

print(expr_matrix_T.shape)

scaler = StandardScaler()
expr_scaled = scaler.fit_transform(expr_matrix_T)

print("Sdandardized data shape:", expr_scaled.shape)
# numpy slicing: [row_range, column_range]

# ============Step 3b: Run PCA and extract results================

# Keep the first 2 principal components for 2D scatter plot
pca = PCA(n_components=2)
pca_result = pca.fit_transform(expr_scaled)

# Build a tidy DataFrame
pca_df = pd.DataFrame(pca_result, columns=['PC1', 'PC2'], index=expr_matrix_T.index)
pca_df['sample'] = pca_df.index
pca_df['condition'] = pca_df['sample'].apply(extract_group)
pca_df['condition'] = pd.Categorical(pca_df['condition'], categories=GROUP_ORDER,ordered=True)

# Variance explained by each PC
var_ration = pca.explained_variance_ratio_ * 100
print(f"PC1 explains {var_ration[0]:.1f}% of variance")
print(f"PC2 explains {var_ration[1]:.1f}% of variance")

pca_df


In [ ]:
# ============step 3b: Filter low-expression genes and redo PCA==========

# Keep gene with KPKM > 1 in at least in 3 samples
expression_mask = (expr_matrix > 1). sum(axis=1) >= 3
expr_filtered =expr_matrix[expression_mask]

print(f"Genes before filtering: {expr_matrix.shape}")
print(f"Genes after filtering: {expr_filtered.shape}")
print(f"Removed {expr_matrix.shape[0] - expr_filtered.shape[0]} low-expression genes")



In [ ]:
# ==============Step 3c: PCA scatter plot==================


fig, ax = plt.subplots(figsize=(7, 6))

sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='condition',
                palette=NATURE_PALETTE, hue_order=GROUP_ORDER,
                s=120, edgecolor='black', linewidth=0.8, ax=ax)

# Annotate each point with sample name
for i in range(len(pca_df)):
    ax.text(pca_df['PC1'].iloc[i] + 5, pca_df['PC2'].iloc[i] + 5,
            pca_df['sample'].iloc[i], fontsize=8)

ax.set_xlabel(f"PC1({var_ration[0]:.1f}%)")
ax.set_ylabel(f"PC2({var_ration[1]:.1f}%)")
ax.set_title('PCA scatter plot')
ax.legend(title='Condition', frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
# ========Step 3d: Standardize filtered data and redo PCA=========

expr_filtered_T = expr_filtered.T     # Transpose: rows=genes -> rows = samples

scaler2 = StandardScaler()
expr_filtered_scaled = scaler2.fit_transform(expr_filtered_T)

pca2 = PCA(n_components=2)
pca_result2 = pca2.fit_transform(expr_filtered_scaled)

pca_df_filtered = pd.DataFrame(pca_result, columns=['PC1', 'PC2'], index=expr_filtered_T.index)
pca_df_filtered['sample'] = pca_df_filtered.index
pca_df_filtered['condition'] = pca_df_filtered['sample'].apply(extract_group)
pca_df_filtered['condition'] = pd.Categorical(pca_df_filtered['condition'], categories=GROUP_ORDER,ordered=True)

var_ratio2 = pca2.explained_variance_ratio_ * 100
print(f"PC1 (filtered) explains {var_ratio2[0]:.1f}% of variance")
print(f"PC2 (filtered) explains {var_ratio2[1]:.1f}% of variance")




In [ ]:
# ============step 3f: PCA scatter plot (filtered genes)==============
fig, ax = plt.subplots(figsize=(7, 6))

sns.scatterplot(data=pca_df_filtered, x='PC1', y='PC2', hue='condition',
                palette=NATURE_PALETTE, hue_order=GROUP_ORDER,
                s=120, edgecolor='black', linewidth=0.8, ax=ax)

# Annotate each point with sample name
for i in range(len(pca_df_filtered)):
    ax.text(pca_df_filtered['PC1'].iloc[i] + 5, pca_df_filtered['PC2'].iloc[i] +5,
            pca_df_filtered['sample'].iloc[i], fontsize=8)

ax.set_xlabel(f"PC1({var_ratio2[0]:.1f}%)")
ax.set_ylabel(f"PC2({var_ratio2[1]:.1f}%)")
ax.set_title('PCA scatter plot(filtered genes)')
ax.legend(title="Condition", frameon=False)
apply_nature_style(ax)  #was missing in Cell 10

plt.tight_layout()
plt.show()

In [ ]:
# ==================Step 3g: log2 transform -> standardize -> PCA===========
#  elementwise log2 on the whole DataFrame; +1 avoids log2(0)
expr_log = np.log2(expr_filtered + 1)

expr_log_T = expr_log.T   #row=samples, columns = genes

scaler3 = StandardScaler()
expr_log_scaled = scaler3.fit_transform(expr_log_T)

pca3 = PCA(n_components=2)
pca_result3 = pca3.fit_transform(expr_log_scaled)

pca_df_log = pd.DataFrame(pca_result3, columns=['PC1', 'PC2'], index=expr_log_T.index)
pca_df_log['sample'] = pca_df_log.index
pca_df_log['condition'] = pca_df_log['sample'].apply(extract_group)
pca_df_log['condition'] = pd.Categorical(pca_df_log['condition'], categories=GROUP_ORDER,ordered=True)

var_ratio3 = pca3.explained_variance_ratio_ * 100
print(f"PC1 (log2) explains {var_ratio3[0]:.1f}% of variance")
print(f"PC2 (log2) explains {var_ratio3[1]:.1f}% of variance")


In [ ]:
# .shape returns (rows, columns); [0] picks the row count = number of genes
print(f"Genes before filtering: {expr_matrix.shape[0]}")
print(f"Genes after filtering: {expr_filtered.shape[0]}")
print(f"Removed: {expr_matrix.shape[0] - expr_filtered.shape[0]} low-expression genes")

In [ ]:
# ==========Step 3h: PCA scatter plot (log2, filtered)==========

# create the figure (canvas) and one drawing area (ax), 7x6 inches
fig, ax = plt.subplots(figsize=(7, 6))

# color points by group; hue_order fixes legend order R, CO, CT, CR
sns.scatterplot(data=pca_df_log, x='PC1', y='PC2', hue='condition',
                palette=NATURE_PALETTE, hue_order=GROUP_ORDER,
                s=120, edgecolor='black', linewidth=0.8, ax=ax)

# write each sample name next to its point (offset +5 to avoid overlap)
for i in range(len(pca_df_log)):
    ax.text(pca_df_log['PC1'].iloc[i] + 5, pca_df_log['PC2'].iloc[i] + 5,
            pca_df_log['sample'].iloc[i], fontsize=8)

# axis labels show how much variance each PC explains
ax.set_xlabel(f"PC1 ({var_ratio3[0]:.1f}%)")
ax.set_ylabel(f"PC2 ({var_ratio3[1]:.1f}%)")
ax.set_title('PCA scatter plot (log2, filtered)')
ax.legend(title='Condition', frameon=False)   # remove legend border
apply_nature_style(ax)                        # your own style helper

plt.tight_layout()   # auto-fix spacing; parentheses are required
plt.show()           # render the figure in Jupyter
# save the current figure to disk; dpi=300 is the usual publication resolution
# bbox_inches='tight' trims extra white margin so labels aren't cut off
fig.savefig('pca_log2_filtered.png', dpi=300, bbox_inches='tight')


In [ ]:
def plot_pca(pca_df, var_ratio, title, ax):
    """Draw a PCA scatter colored by condition on the given axes."""
    sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='condition',
                    hue_order=GROUP_ORDER, palette=NATURE_PALETTE,
                    s=120, edgecolor='black', linewidth=0.8, ax=ax)

    # label each point; offset is in points, so it is independent of axis scale
    for x, y, name in zip(pca_df['PC1'], pca_df['PC2'], pca_df['sample']):
        ax.annotate(name, (x, y), xytext=(5, 5), textcoords='offset points',
                    fontsize=8, color='black')

    ax.set_xlabel(f'PC1 ({var_ratio[0]:.1f}%)', color='black')
    ax.set_ylabel(f'PC2 ({var_ratio[1]:.1f}%)', color='black')
    ax.set_title(title, fontsize=10, color='black')

    legend = ax.legend(title=' ', frameon=False, labelcolor='black')
    legend.get_title().set_color('black')   # legend title is a separate object
    apply_nature_style(ax)


fig, ax = plt.subplots(figsize=(7, 6))
plot_pca(pca_df_log, var_ratio3, 'PCA', ax)

plt.tight_layout()
save_figure(fig, r"D:\Python-learning\week02\figures\fig3_pca.png")   # save before show
plt.show()